# NumPy: A Hands-On Lesson

NumPy is the foundation of numerical Python. Pandas, scikit-learn, and most data-science libraries are built on top of it.

This lesson goes in order:

1. Creating arrays
2. Shape, dimensions, and dtypes
3. Indexing and slicing
4. Boolean masks and filtering
5. Vectorized math and broadcasting
6. Aggregations (`sum`, `mean`, and `axis`)
7. Reshaping and stacking
8. Random numbers
9. Why NumPy is fast
10. Practice exercises, with solutions

Run the cells in order. Try changing the values and re-running to see what happens.

## 1. Setup and creating arrays

We import NumPy under the conventional alias `np`.

In [6]:
import numpy as np

print(np.__version__)

2.5.4


The main object is the **`ndarray`**, a grid of values that all share one type. You can build one from a Python list:

In [7]:
a = np.array([3, 7, 1, 9])

print(a)
print(type(a))

[3 7 1 9]
<class 'numpy.ndarray'>


Other helpers create arrays without typing every value:

In [8]:
print("zeros:   ", np.zeros(5))
print("ones:    ", np.ones(3))
print("range:   ", np.arange(0, 10, 2))      # start, stop (exclusive), step
print("linspace:", np.linspace(0, 1, 5))     # 5 evenly spaced points from 0 to 1
print("identity:\n", np.eye(3))

zeros:    [0. 0. 0. 0. 0.]
ones:     [1. 1. 1.]
range:    [0 2 4 6 8]
linspace: [0.   0.25 0.5  0.75 1.  ]
identity:
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


Note that `arange` excludes the stop value, while `linspace` includes it by default.

## 2. Shape, dimensions, and dtype

Every array has a **shape** (its size along each axis), a **number of dimensions** (`ndim`), and a **dtype** (the type of each element).

In [9]:
x = np.array([[1, 2, 3],
              [4, 5, 6]])

print("array:\n", x)
print("shape:", x.shape)     # (rows, columns)
print("ndim: ", x.ndim)
print("size: ", x.size)     # total number of elements
print("dtype:", x.dtype)

array:
 [[1 2 3]
 [4 5 6]]
shape: (2, 3)
ndim:  2
size:  6
dtype: int64


Unlike Python lists, every element of an array has the same dtype. You can choose it when you create the array. Choosing a smaller dtype saves memory:

In [10]:
f = np.array([1, 2, 3], dtype=np.float64)
small = np.array([1, 2, 3], dtype=np.int8)

print(f, f.dtype)
print(small, small.dtype)
print("bytes per element:", f.itemsize, "vs", small.itemsize)

[1. 2. 3.] float64
[1 2 3] int8
bytes per element: 8 vs 1


Converting with `astype` creates a new array. Be careful with float-to-int conversion, since it truncates decimals:

In [11]:
prices = np.array([10.7, 3.2, 8.9])
print(prices.astype(int))  # truncates toward zero

[10  3  8]


## 3. Indexing and slicing

Indexing works like Python lists, but with one index per axis. Slicing uses `start:stop:step`, and `stop` is excluded.

In [12]:
v = np.array([10, 20, 30, 40, 50])

print("first:      ", v[0])
print("last:       ", v[-1])
print("slice 1:3:  ", v[1:3])     # elements at index 1 and 2
print("every other:", v[::2])
print("reversed:   ", v[::-1])

first:       10
last:        50
slice 1:3:   [20 30]
every other: [10 30 50]
reversed:    [50 40 30 20 10]


Slices are **views**: they share memory with the original array. Changing a slice changes the original. Use `.copy()` when you want an independent array:

In [13]:
view = v[1:3]
view[0] = 999
print("after editing the view:", v)

v = np.array([10, 20, 30, 40, 50])   # reset
independent = v[1:3].copy()
independent[0] = 999
print("after editing the copy:", v)

after editing the view: [ 10 999  30  40  50]
after editing the copy: [10 20 30 40 50]


For 2D arrays, use `[row, column]`:

In [14]:
m = np.arange(12).reshape(3, 4)
print(m)
print()
print("element at row 1, col 2:", m[1, 2])
print("whole row 0:           ", m[0, :])
print("column 3:              ", m[:, 3])
print("top-left 2x2 block:\n", m[:2, :2])

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]

element at row 1, col 2: 6
whole row 0:            [0 1 2 3]
column 3:               [ 3  7 11]
top-left 2x2 block:
 [[0 1]
 [4 5]]


## 4. Boolean masks and filtering

A comparison on an array returns an array of `True`/`False`. Using that mask as an index selects the matching elements. This is one of the most-used NumPy patterns.

In [15]:
claims = np.array([120, 45, 890, 230, 15, 640])

mask = claims > 200
print("mask:    ", mask)
print("expensive:", claims[mask])
print("count over 200:", np.sum(mask))   # True counts as 1

mask:     [False False  True  True False  True]
expensive: [890 230 640]
count over 200: 3


You can combine conditions with `&` (and), `|` (or), and `~` (not). Parentheses are required because of operator precedence:

In [16]:
print(claims[(claims > 50) & (claims < 500)])
print(claims[(claims < 50) | (claims > 600)])
print(claims[~(claims > 200)])

[120 230]
[ 45 890  15 640]
[120  45  15]


`np.where` returns values based on a condition. It works like an if/else applied to every element:

In [17]:
flagged = np.where(claims > 200, "high", "low")
print(flagged)

['low' 'low' 'high' 'high' 'low' 'high']


## 5. Vectorized math and broadcasting

Arithmetic applies to every element at once. Explicit loops are rarely needed:

In [18]:
costs = np.array([100.0, 250.0, 80.0])

print("add 10:      ", costs + 10)
print("double:      ", costs * 2)
print("percent up:  ", costs * 1.05)
print("square:      ", costs ** 2)
print("log:         ", np.log(costs))

add 10:       [110. 260.  90.]
double:       [200. 500. 160.]
percent up:   [105.  262.5  84. ]
square:       [10000. 62500.  6400.]
log:          [4.60517019 5.52146092 4.38202663]


Arithmetic between two arrays works element by element, as long as the shapes match:

In [19]:
a = np.array([1, 2, 3])
b = np.array([10, 20, 30])
print(a * b)

[10 40 90]


**Broadcasting** lets NumPy combine arrays of different shapes. The smaller array is "stretched" to match the larger one. A common case is adding a single value or a row vector to every row of a matrix:

In [20]:
matrix = np.array([[1, 2, 3],
                   [4, 5, 6]])
per_column_offset = np.array([100, 200, 300])

print(matrix + per_column_offset)     # adds the row vector to each row
print(matrix + 1000)                  # adds a scalar to every element

[[101 202 303]
 [104 205 306]]
[[1001 1002 1003]
 [1004 1005 1006]]


Broadcasting rules: shapes are compared from the right. Each pair of dimensions must be equal, or one of them must be 1. The example below shows a column vector being broadcast across a matrix:

In [21]:
column = np.array([[1], [10]])        # shape (2, 1)
row = np.array([1, 2, 3])             # shape (3,)
print(column + row)                   # shape (2, 3)

[[ 2  3  4]
 [11 12 13]]


## 6. Aggregations and `axis`

Functions like `sum`, `mean`, `min`, `max`, and `std` summarize an array. Without an `axis`, they reduce everything to one number. With an `axis`, they reduce along one dimension.

In [22]:
data = np.array([[4, 8, 6],
                 [1, 5, 9]])

print("total sum:    ", data.sum())
print("mean:         ", data.mean())
print("sum per column (axis=0):", data.sum(axis=0))
print("sum per row    (axis=1):", data.sum(axis=1))
print("max per row:  ", data.max(axis=1))

total sum:     33
mean:          5.5
sum per column (axis=0): [ 5 13 15]
sum per row    (axis=1): [18 15]
max per row:   [8 9]


A handy way to remember `axis`: `axis=0` collapses the **rows** (going down), and `axis=1` collapses the **columns** (going across).

Here is a realistic example. Suppose each row is a patient and each column is a month of claim cost:

In [23]:
monthly = np.array([
    [120, 90, 300],   # patient 1
    [ 40, 45,  50],   # patient 2
    [800, 60, 75],    # patient 3
])

print("total per patient:", monthly.sum(axis=1))
print("average per month:", monthly.mean(axis=0).round(1))
print("highest month per patient:", monthly.max(axis=1))

total per patient: [510 135 935]
average per month: [320.   65.  141.7]
highest month per patient: [300  50 800]


Other useful functions: `np.percentile`, `np.median`, `np.argmax` (index of the maximum), and `np.cumsum` (running total).

In [24]:
print("median:", np.median(monthly))
print("90th percentile:", np.percentile(monthly, 90))
print("index of largest value (flattened):", np.argmax(monthly))
print("running total:", np.cumsum([1, 2, 3, 4]))

median: 75.0
90th percentile: 400.0000000000001
index of largest value (flattened): 6
running total: [ 1  3  6 10]


## 7. Reshaping and stacking

In [25]:
r = np.arange(12)
print("original:", r.shape)

print(r.reshape(3, 4))
print(r.reshape(2, 2, 3).shape)
print(r.reshape(3, -1).shape)    # -1 means "work it out automatically"

original: (12,)
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
(2, 2, 3)
(3, 4)


`reshape` changes only the view of the data, not the values themselves. The total number of elements must stay the same.

Combining arrays:

In [26]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

print("vstack (stack rows):\n", np.vstack([a, b]))
print("hstack (join end to end):", np.hstack([a, b]))
print("column_stack:\n", np.column_stack([a, b]))

vstack (stack rows):
 [[1 2 3]
 [4 5 6]]
hstack (join end to end): [1 2 3 4 5 6]
column_stack:
 [[1 4]
 [2 5]
 [3 6]]


Transposing swaps rows and columns:

In [27]:
t = np.array([[1, 2, 3],
              [4, 5, 6]])
print(t.T)
print(t.T.shape)

[[1 4]
 [2 5]
 [3 6]]
(3, 2)


## 8. Random numbers

NumPy's `random` module generates random data. The modern approach uses a generator object. Passing a `seed` makes results reproducible, which matters when you're comparing models or writing tests.

In [28]:
rng = np.random.default_rng(seed=42)

samples = rng.normal(loc=100, scale=15, size=5)   # mean 100, standard deviation 15
print("normal samples:", samples.round(2))

print("integers 0-9:  ", rng.integers(0, 10, size=8))
print("shuffled:      ", rng.permutation(np.arange(6)))

normal samples: [104.57  84.4  111.26 114.11  70.73]
integers 0-9:   [5 9 7 7 7 7 5 1]
shuffled:       [4 0 2 3 5 1]


Running the cell again with the same seed gives the same numbers. Try it.

## 9. Why NumPy is fast

A Python loop runs one step at a time through the interpreter. NumPy runs the same operation in compiled C code over the whole array. Let's compare them:

In [29]:
import time

n = 1_000_000
python_list = list(range(n))
numpy_array = np.arange(n)

start = time.perf_counter()
total = 0
for value in python_list:
    total += value * 2
loop_time = time.perf_counter() - start

start = time.perf_counter()
total_np = (numpy_array * 2).sum()
numpy_time = time.perf_counter() - start


print(f"Python loop: {loop_time:.4f} s")
print(f"NumPy:       {numpy_time:.4f} s")
print(f"Both totals match: {total == total_np}")

Python loop: 0.2492 s
NumPy:       0.0118 s
Both totals match: True


Expect NumPy to be many times faster. The exact speed depends on your machine. The rule of thumb is to **avoid Python loops over array elements** and write the operation as a whole-array expression.

## 10. Practice exercises

Try each exercise before you look at the solution. Cells with `# TODO` are for you to fill in.

**Exercise 1.** Create an array of the integers 1 to 20. Print only the even numbers.

In [35]:
nums = np.arange(1,21)
print(nums[nums % 2 == 0])

[ 2  4  6  8 10 12 14 16 18 20]


<details><summary>Solution</summary>

```python
nums = np.arange(1, 21)
print(nums[nums % 2 == 0])
```

</details>

**Exercise 2.** Given the array of claim amounts below, compute the mean of the claims over 500, and the count of claims at or under 100.

In [41]:
claims = np.array([55, 620, 130, 980, 75, 410, 1200, 95])

print(claims[claims > 500].mean())
print(claims[claims <= 100].size)
print(np.sum(claims <= 100))

933.3333333333334
3
3


<details><summary>Solution</summary>

```python
print("mean over 500:", claims[claims > 500].mean())
print("count <= 100:", np.sum(claims <= 100))
```

</details>

**Exercise 3.** A matrix has one row per patient and one column per month. Compute the average monthly cost for each month, and the number of months each patient spent above 200.

In [50]:
costs = np.array([
    [150, 320, 80],
    [ 40, 210, 900],
    [ 60,  75, 110],
    [500, 260, 230],
])

print(costs.mean(axis=0).round(2))
print((costs > 200).sum(axis=1))

[187.5  216.25 330.  ]
[1 2 0 3]


<details><summary>Solution</summary>

```python
print("average per month:", costs.mean(axis=0))
print("months above 200 per patient:", (costs > 200).sum(axis=1))
```

</details>

**Exercise 4.** Turn the flat array below into a 4 x 3 matrix, then scale each column so its values are relative to that column's maximum (divide each column by its own max).

In [61]:
flat = np.arange(1, 13)
m = flat.reshape(4,3)
print(m / m.max(axis=0))


[[0.1        0.18181818 0.25      ]
 [0.4        0.45454545 0.5       ]
 [0.7        0.72727273 0.75      ]
 [1.         1.         1.        ]]


<details><summary>Solution</summary>

```python
m = flat.reshape(4, 3)
print(m / m.max(axis=0))
```

</details>

## Summary

- **Arrays** hold values of one dtype, with a fixed shape.
- **Indexing and slicing** select parts of an array. Slices are views, so use `.copy()` if you need independence.
- **Boolean masks** filter data by condition.
- **Vectorized operations and broadcasting** replace explicit loops.
- **`axis`** controls which dimension an aggregation collapses.
- **`reshape` and stacking** rearrange data without changing the values.
- **Seeded random generators** make results reproducible.

Next, a good step is pandas, which is built on NumPy arrays.

In [1]:
x = 6